# Phase 3 / p3_04 -- analysis: figures + McNemar significance tests

Reads `results.csv` + `curves.csv` (and the pooled per-item prediction files
`results/preds/<run_id>__noise_s{3,5}*.csv`) **only** -- never trains, never
runs inference, CPU-only, fast enough to run locally or on a CPU-only Kaggle
session (no GPU quota spent).

Two outputs:
1. All 8 figure types (`pipeline/p3_figures.py`) -- `.png`/`.pdf`/`.csv` each.
2. The 6 pre-registered McNemar tests (METHODOLOGY M6.2): BanglaBERT vs
   char_ngram, at severity 3 and severity 5, for each of the 3 tasks,
   Holm-corrected across all 6. **Only these 6** are ever reported as
   "significant" / "not significant" -- everything else in the figures is
   descriptive.

In [ ]:
import os, sys, shutil, time

# ---------------------------------------------------------------------------
# Kaggle bootstrap. Before running this notebook on Kaggle, attach THREE
# private Datasets via the "Add Data" panel (see PHASE3_STATUS.md for exact
# creation steps):
#   1. bangla-noisebench-data-final -- data/final/ (SentNoB is CC-BY-ND-4.0,
#      this Dataset MUST be private)
#   2. bangla-noisebench-results    -- the versioned results.csv/curves.csv/
#      preds/ state, so a killed session can resume for free
#   3. bangla-noisebench-code       -- this repository, so `pipeline` and
#      `noisebench` are importable (Kaggle notebooks don't see the repo
#      checkout directly)
# Change the three slugs below if yours differ.
# ---------------------------------------------------------------------------
DATA_DATASET_SLUG = "bangla-noisebench-data-final"
RESULTS_DATASET_SLUG = "bangla-noisebench-results"
CODE_DATASET_SLUG = "bangla-noisebench-code"

IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    CODE_ROOT = f"/kaggle/input/{CODE_DATASET_SLUG}/bangla-noisebench"
    sys.path.insert(0, CODE_ROOT)
    DATA_FINAL_ROOT = f"/kaggle/input/{DATA_DATASET_SLUG}/final"
    RESULTS_SRC = f"/kaggle/input/{RESULTS_DATASET_SLUG}"
    RESULTS_DIR = "/kaggle/working/results"
    os.makedirs(RESULTS_DIR, exist_ok=True)
    for fname in ("results.csv", "curves.csv"):
        src = os.path.join(RESULTS_SRC, fname)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(RESULTS_DIR, fname))
            print(f"[bootstrap] restored {fname} from {RESULTS_SRC}")
        else:
            print(f"[bootstrap] no prior {fname} in {RESULTS_SRC} -- starting fresh")
    preds_src = os.path.join(RESULTS_SRC, "preds")
    if os.path.isdir(preds_src):
        shutil.copytree(preds_src, os.path.join(RESULTS_DIR, "preds"), dirs_exist_ok=True)
        print(f"[bootstrap] restored preds/ ({len(os.listdir(os.path.join(RESULTS_DIR, 'preds')))} files)")
else:
    _ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    sys.path.insert(0, _ROOT)
    DATA_FINAL_ROOT = os.path.join(_ROOT, "data", "final")
    RESULTS_DIR = os.path.join(_ROOT, "results")
    print(f"[bootstrap] not on Kaggle -- using local repo paths under {_ROOT}")

print(f"DATA_FINAL_ROOT = {DATA_FINAL_ROOT}")
print(f"RESULTS_DIR     = {RESULTS_DIR}")


In [ ]:
# figures don't need torch/transformers -- only the packages already on any
# Kaggle image (pandas/numpy/matplotlib) plus statsmodels for McNemar.
%pip install -q "statsmodels>=0.14"


In [ ]:
from pipeline import p3_train, p3_figures

progress = p3_train.Progress("notebook")
p3_figures.build_all(RESULTS_DIR, os.path.join(RESULTS_DIR, "..", "outputs", "p3_figures"),
                     p3_figures._DEFAULT_CER_TABLE, progress)


In [ ]:
# ---------------------------------------------------------------------------
# M6.2 -- McNemar's test, BanglaBERT vs char_ngram, severity in {3, 5}, one
# test per task (3 tasks x 2 severities = 6 tests total), Holm-corrected.
#
# Design decision (PHASE3_STATUS.md): each test pools the 3 seeds' AND all 9
# noise types' per-item predictions for that (task, severity) into one 2x2
# contingency table (each (test-set item, noise type, seed) triple
# contributes one paired observation -- the pooled preds file has 9 noise
# types' worth of rows per item at a fixed severity, see
# pipeline/p3_evaluate_noise.py's pooling comment). This is a common
# simplification for reporting a single model-vs-model test across seeds and
# noise types; it slightly understates the effective sample size
# (observations are not fully independent) but never overstates -- p-values
# here are conservative, not anti-conservative.
#
# IMPORTANT: the merge below is on ["id", "noise_type"], NOT "id" alone --
# each pooled file repeats every test-set id once per noise type, so an
# id-only merge would cross-join banglabert's char_insert prediction against
# char_ngram's char_delete prediction for the "same" id and silently produce
# a nonsensical (and much larger) contingency table. Caught in this repo's
# smoke test (a real, if tiny, McNemar run) before it ever reached real data.
# ---------------------------------------------------------------------------
import glob
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

from pipeline.p3_train import TASK_ORDER, SEEDS
from pipeline.p3_evaluate_noise import MCNEMAR_SEVERITIES

preds_dir = os.path.join(RESULTS_DIR, "preds")
rows = []
for task in TASK_ORDER:
    for severity in MCNEMAR_SEVERITIES:
        both_correct = both_wrong = a_only = b_only = 0
        n_seed_files = 0
        for seed in SEEDS:
            a_path = os.path.join(preds_dir, f"banglabert__{task}__{seed}__clean__noise_s{severity}.csv")
            b_path = os.path.join(preds_dir, f"char_ngram__{task}__{seed}__clean__noise_s{severity}.csv")
            if not (os.path.exists(a_path) and os.path.exists(b_path)):
                continue
            n_seed_files += 1
            a = pd.read_csv(a_path)
            b = pd.read_csv(b_path)
            merged = a.merge(b, on=["id", "noise_type"], suffixes=("_a", "_b"))
            a_correct = merged.true_a == merged.pred_a
            b_correct = merged.true_b == merged.pred_b
            both_correct += int((a_correct & b_correct).sum())
            both_wrong += int((~a_correct & ~b_correct).sum())
            a_only += int((a_correct & ~b_correct).sum())
            b_only += int((~a_correct & b_correct).sum())
        if n_seed_files == 0:
            print(f"SKIP {task} sev={severity}: no pooled prediction files found yet")
            continue
        table = [[both_correct, a_only], [b_only, both_wrong]]
        result = mcnemar(table, exact=(a_only + b_only) < 25, correction=True)
        rows.append({"task": task, "severity": severity, "n_seed_files": n_seed_files,
                    "both_correct": both_correct, "banglabert_only_correct": a_only,
                    "char_ngram_only_correct": b_only, "both_wrong": both_wrong,
                    "statistic": result.statistic, "p_raw": result.pvalue})

mcnemar_df = pd.DataFrame(rows)
if not mcnemar_df.empty:
    reject, p_holm, _, _ = multipletests(mcnemar_df.p_raw, alpha=0.05, method="holm")
    mcnemar_df["p_holm"] = p_holm
    mcnemar_df["significant_holm_0.05"] = reject
    print(mcnemar_df.to_string(index=False))
    out_path = os.path.join(RESULTS_DIR, "..", "outputs", "p3_figures", "mcnemar_m6_2.csv")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    mcnemar_df.to_csv(out_path, index=False)
    print(f"saved {out_path}")
else:
    print("No McNemar tests could be run yet -- need banglabert + char_ngram "
         "pooled prediction files for at least one (task, severity in {3,5}) cell. "
         "Run p3_01/p3_02 for both models first.")
